<a href="https://colab.research.google.com/github/Ameer-pasha/gpt-oss-text2sql-lora/blob/main/nb/gpt-oss-(20B)-Fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [2]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.08 GB.


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Detected MoE model with per-expert Linear experts. Enabling LoRA on 64 expert projection modules.


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [4]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "Solve x^5 + 3x^4 - 10 = 3." Wait maybe it's an equation: x^5 + 3x^4 - 10 = 3. The variable x unknown. Solve for x. We need to solve the equation:

x^


Lastly we will test it using `reasoning_effort` to `high`

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the equation: x^5 + 3x^4 - 10 = 3. Or maybe it's x^5 + 3x^4 - 10 = 3? That seems like a polynomial equation: x^5 + 3x^4 - 10


<a name="Data"></a>
### Data Prep

The `HuggingFaceH4/Multilingual-Thinking` dataset will be utilized as our example. This dataset, available on Hugging Face, contains reasoning chain-of-thought examples derived from user questions that have been translated from English into four other languages. It is also the same dataset referenced in OpenAI's [cookbook](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers) for fine-tuning. The purpose of using this dataset is to enable the model to learn and develop reasoning capabilities in these four distinct languages.

In [5]:
from datasets import load_dataset

raw = load_dataset(
    "gretelai/synthetic_text_to_sql",
    split="train"
)

print(raw.column_names)
print(raw[0])

['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']
{'id': 5097, 'domain': 'forestry', 'domain_description': 'Comprehensive data on sustainable forest management, timber production, wildlife habitat, and carbon sequestration in forestry.', 'sql_complexity': 'single join', 'sql_complexity_description': 'only one join (specify inner, outer, cross)', 'sql_task_type': 'analytics and reporting', 'sql_task_type_description': 'generating reports, dashboards, and analytical insights', 'sql_prompt': 'What is the total volume of timber sold by each salesperson, sorted by salesperson?', 'sql_context': "CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale

In [6]:
from datasets import load_dataset
import sqlite3

raw = load_dataset("gretelai/synthetic_text_to_sql", split="train")
print(sorted(raw.unique("domain")))          # 100 domains ki list

industry = ["manufacturing", "automotive", "chemicals", "mining operations",
            "civil engineering", "construction", "energy", "maritime",
            "waste management", "food industry"]

def pre_ok(row):
    return (
        row["domain"] in industry
        and row["sql_task_type"] == "analytics and reporting"
        and len(row["sql_context"]) <= 900
    )

cand = raw.filter(pre_ok)
print("domain filter ke baad:", len(cand))

def runs_ok(row):
    try:
        con = sqlite3.connect(":memory:")
        con.executescript(row["sql_context"])
        rows = con.execute(row["sql"]).fetchall()
        con.close()
        return len(rows) > 0
    except Exception:
        return False

clean = cand.filter(runs_ok).shuffle(seed=42)
print("SQL chalne ke baad:", len(clean))
print(clean[0]["domain"], "|", clean[0]["sql_prompt"])

['aerospace', 'agriculture', 'aquaculture', 'archeology', 'arctic research', 'artificial intelligence', 'arts and culture', 'arts operations and management', 'automotive', 'beauty industry', 'biotechnology', 'blockchain', 'cannabis industry', 'charitable organizations', 'chemicals', 'civil engineering', 'climate change', 'construction', 'cosmetics', 'cultural preservation', 'cybersecurity', 'defense contractors', 'defense industry', 'defense operations', 'defense security', 'disability services', 'education', 'energy', 'entertainment industry', 'ethical fashion', 'fashion', 'fashion retail', 'finance', 'financial services', 'fine arts', 'fitness industry', 'food industry', 'food services', 'forestry', 'gaming industry', 'gaming technology', 'government policy', 'government services', 'healthcare', 'higher education', 'hospitality', 'hospitality technology', 'human resources', 'humanitarian aid', 'insurance', 'journalism', 'justice', 'legal services', 'logistics', 'manufacturing', 'mari

To format our dataset, we will apply our version of the GPT OSS prompt

In [7]:
SYSTEM = "You are a SQL expert. Given a database schema and a question, reply with only the SQL query."

def to_messages(row):
    user = f"Database schema:\n{row['sql_context']}\n\nQuestion: {row['sql_prompt']}"
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": user},
        {"role": "assistant",
         "thinking": row["sql_explanation"],   # NAYA: soch (analysis channel)
         "content":  row["sql"]},              # jawab (final channel)
    ]}

train_ds = (clean.select(range(min(1500, len(clean))))
                 .map(to_messages, remove_columns=clean.column_names))

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize = False, add_generation_prompt = False)
             for c in convos]
    return { "text" : texts, }

dataset = train_ds.map(formatting_prompts_func, batched = True)

lens = [len(tokenizer(t)["input_ids"]) for t in dataset["text"]]
dataset = dataset.select([i for i, l in enumerate(lens) if l <= 1000])
print("final rows:", len(dataset))

print(dataset[0]["text"])
assert "<|start|>assistant<|channel|>final<|message|>" in dataset[0]["text"]

final rows: 1500
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-09-20

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a SQL expert. Given a database schema and a question, reply with only the SQL query.<|end|><|start|>user<|message|>Database schema:
CREATE TABLE project_timeline (project_id INT, project_name VARCHAR(30), duration INT); INSERT INTO project_timeline (project_id, project_name, duration) VALUES (1, 'Sustainable Project A', 240), (2, 'Traditional Project A', 120), (3, 'Sustainable Project B', 300); CREATE TABLE project (project_id INT, state VARCHAR(20), type VARCHAR(20)); INSERT INTO project (project_id, state, type) VALUES (1, 'California', 'Sustainable'), (2, 'New York', 'Traditional'), (3, 'California

Let's take a look at the dataset, and check what the 1st example shows

In [8]:
print(dataset[0]['text'])

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-09-20

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a SQL expert. Given a database schema and a question, reply with only the SQL query.<|end|><|start|>user<|message|>Database schema:
CREATE TABLE project_timeline (project_id INT, project_name VARCHAR(30), duration INT); INSERT INTO project_timeline (project_id, project_name, duration) VALUES (1, 'Sustainable Project A', 240), (2, 'Traditional Project A', 120), (3, 'Sustainable Project B', 300); CREATE TABLE project (project_id INT, state VARCHAR(20), type VARCHAR(20)); INSERT INTO project (project_id, state, type) VALUES (1, 'California', 'Sustainable'), (2, 'New York', 'Traditional'), (3, 'California', 'Sustainable')

What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [8]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes and lower loss as well!

In [12]:
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(instruction_part = "<|start|>user<|message|>", response_part = "<|start|>assistant<|channel|>final<|message|>")

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [13]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-09-20\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions\n\nYou are a SQL expert. Given a database schema and a question, reply with only the SQL query.<|end|><|start|>user<|message|>Database schema:\nCREATE TABLE products (id INT, name VARCHAR(255), serving_size INT, protein_grams FLOAT); INSERT INTO products (id, name, serving_size, protein_grams) VALUES (1, 'Product H', 250, 8.0); INSERT INTO products (id, name, serving_size, protein_grams) VALUES (2, 'Product I', 150, 5.0); INSERT INTO products (id, name, serving_size, protein_grams) VALUES (3, 'Product J', 300, 12.0);\n\nQuestion: How many grams of protein are in products with a serving size larger than 200?<|end|><|sta

Now let's print the masked out example - you should see only the answer is present:

In [14]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                                                                                   SELECT SUM(protein_grams) AS total_protein_grams FROM products WHERE serving_size > 200;<|return|>'

In [13]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
12.771 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [13]:
import torch, os
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0), "| count:", torch.cuda.device_count())
print("env:", {k: os.environ.get(k) for k in ["UNSLOTH_COMPILE_DISABLE", "TORCHDYNAMO_DISABLE", "CUDA_VISIBLE_DEVICES"]})
print("trainable dtypes:", {p.dtype for n, p in model.named_parameters() if p.requires_grad})
print("sinks dtypes:", {p.dtype for n, p in model.named_parameters() if "sinks" in n})

GPU: Tesla T4 | capability: (7, 5) | count: 1
env: {'UNSLOTH_COMPILE_DISABLE': '1', 'TORCHDYNAMO_DISABLE': '1', 'CUDA_VISIBLE_DEVICES': '0'}
trainable dtypes: {torch.float32}
sinks dtypes: {torch.float16}


In [16]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [18]:
import transformers.models.gpt_oss.modeling_gpt_oss as gpt_oss_mod

if not hasattr(gpt_oss_mod, "_orig_eager_attention_forward"):
    gpt_oss_mod._orig_eager_attention_forward = gpt_oss_mod.eager_attention_forward

_seen = {"done": False}

def patched_eager_attention_forward(module, query, key, value, *args, **kwargs):
    if not _seen["done"]:
        print("q/k/v dtypes:", query.dtype, key.dtype, value.dtype, "| sinks:", module.sinks.dtype)
        _seen["done"] = True
    return gpt_oss_mod._orig_eager_attention_forward(
        module, query.float(), key.float(), value.float(), *args, **kwargs
    )

gpt_oss_mod.eager_attention_forward = patched_eager_attention_forward
print("patched")

patched


In [19]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 1,500 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 92,454,912 of 21,007,212,096 (0.44% trained)


q/k/v dtypes: torch.float16 torch.float16 torch.float16 | sinks: torch.float16


Step,Training Loss
1,0.726400
2,0.781500
3,0.658800
4,0.347500
5,0.100900
6,0.021100
7,0.057300
8,0.217900
9,0.146000
10,0.124700


In [24]:
schema = "CREATE TABLE machines (id INT, name TEXT, line TEXT); CREATE TABLE breakdowns (id INT, machine_id INT, downtime_min INT, date DATE);"
question = "Which machine had the most total downtime in 2024?"

In [25]:
def ask_sql_final(schema, question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Database schema:\n{schema}\n\nQuestion: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = False,                  # abhi text chahiye, numbers nahi
        reasoning_effort = "medium",       # training text mein bhi medium tha
    )
    prompt += "<|channel|>final<|message|>"    # seedha jawab wale channel se shuru

    inputs = tokenizer(prompt, return_tensors = "pt", add_special_tokens = False).to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens = False)
    return text.split("<|")[0].strip()         # <|return|> jaise marker se pehle ka hissa

print("AFTER :", ask_sql_final(schema, question))
print()
with model.disable_adapter():
    print("BEFORE:", ask_sql_final(schema, question))

AFTER : SELECT machine_id, SUM(downtime_min) AS total_downtime FROM breakdowns WHERE date BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY machine_id ORDER BY total_downtime DESC LIMIT 1;

BEFORE: SELECT m.name
FROM machines m
JOIN breakdowns d
  ON m.id = d.machine_id
WHERE d.date >= '2024-01-01' AND d.date <= '2024-12-31'
GROUP BY m.id, m.name
ORDER BY SUM(d.downtime_min) DESC
LIMIT 1;


In [26]:
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(
    instruction_part = "<|start|>user<|message|>",
    response_part    = "<|start|>assistant<|channel|>analysis<|message|>",   # analysis + final dono
)

trainer = train_on_responses_only(trainer, **gpt_oss_kwargs)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

645.6936 seconds used for training.
10.76 minutes used for training.
Peak reserved memory = 12.975 GB.
Peak reserved memory for training = 0.164 GB.
Peak reserved memory % of max memory = 88.02 %.
Peak reserved memory for training % of max memory = 1.113 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [28]:
from transformers import TextStreamer

SYSTEM = "You are a SQL expert. Given a database schema and a question, reply with only the SQL query."

def ask_sql(schema, question, effort="medium", max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Database schema:\n{schema}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,     # "ab assistant jawab de" ka tag jodta hai
        return_tensors = "pt",
        return_dict = True,
        reasoning_effort = effort,
    ).to("cuda")                           # inputs GPU par bhejta hai
    _ = model.generate(**inputs, max_new_tokens = max_new_tokens, streamer = TextStreamer(tokenizer))

schema = "CREATE TABLE machines (id INT, name TEXT, line TEXT); CREATE TABLE breakdowns (id INT, machine_id INT, downtime_min INT, date DATE);"
question = "Which machine had the most total downtime in 2024?"

print("=== AFTER (fine-tuned) ===")
ask_sql(schema, question)

=== AFTER (fine-tuned) ===
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-09-20

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a SQL expert. Given a database schema and a question, reply with only the SQL query.<|end|><|start|>user<|message|>Database schema:
CREATE TABLE machines (id INT, name TEXT, line TEXT); CREATE TABLE breakdowns (id INT, machine_id INT, downtime_min INT, date DATE);

Question: Which machine had the most total downtime in 2024?<|end|><|start|>assistant<|channel|>analysis<|message|>The user has provided a database schema with two tables: machines and breakdowns. The question is about the machine that had the most total downtime in 2024, which presumably means we need to query the breakdowns tabl

In [29]:
def ask_sql_final(schema, question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Database schema:\n{schema}\n\nQuestion: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = False,
        reasoning_effort = "medium",
    )
    prompt += "<|channel|>final<|message|>"      # seedha jawab wale channel se shuru

    inputs = tokenizer(prompt, return_tensors = "pt", add_special_tokens = False).to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens = False)
    return text.split("<|")[0].strip()

print("AFTER :", ask_sql_final(schema, question))
print()
with model.disable_adapter():
    print("BEFORE:", ask_sql_final(schema, question))

AFTER : SELECT name FROM machines WHERE id = (SELECT machine_id FROM breakdowns WHERE year(date)=2024 GROUP BY machine_id ORDER BY SUM(downtime_min) DESC LIMIT 1);

BEFORE: SELECT 
    m.id,
    m.name,
    SUM(b.downtime_min) AS total_downtime
FROM machines m
JOIN breakdowns b ON m.id = b.machine_id
WHERE YEAR(b.date) = 2024
GROUP BY m.id, m.name
ORDER BY total_downtime DESC
LIMIT 1;


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** Currently finetunes can only be loaded via Unsloth in the meantime - we're working on vLLM and GGUF exporting!

In [31]:
print("trainer_stats hai:", "trainer_stats" in globals())
if "trainer_stats" in globals():
    print(trainer_stats.metrics)
!ls -la /content/outputs
!ls -la /content/outputs/*

trainer_stats hai: True
{'train_runtime': 536.1197, 'train_samples_per_second': 0.224, 'train_steps_per_second': 0.056, 'total_flos': 4302153364953600.0, 'train_loss': 0.21085427701473236, 'epoch': 0.08}
total 16
drwxr-xr-x 3 root root 4096 Sep 20 11:32 .
drwxr-xr-x 1 root root 4096 Sep 20 11:18 ..
drwxr-xr-x 2 root root 4096 Sep 20 11:32 checkpoint-30
-rw-r--r-- 1 root root 1481 Sep 20 11:32 README.md
-rw-r--r-- 1 root root 1481 Sep 20 11:32 /content/outputs/README.md

/content/outputs/checkpoint-30:
total 574232
drwxr-xr-x 2 root root      4096 Sep 20 11:32 .
drwxr-xr-x 3 root root      4096 Sep 20 11:32 ..
-rw-r--r-- 1 root root      1585 Sep 20 11:32 adapter_config.json
-rw------- 1 root root 370291488 Sep 20 11:32 adapter_model.safetensors
-rw-r--r-- 1 root root     15078 Sep 20 11:32 chat_template.jinja
-rw-r--r-- 1 root root 189762167 Sep 20 11:32 optimizer.pt
-rw-r--r-- 1 root root      5248 Sep 20 11:32 README.md
-rw-r--r-- 1 root root     14645 Sep 20 11:32 rng_state.pth
-rw-

In [33]:
from google.colab import userdata
from huggingface_hub import upload_folder

hf_token = userdata.get("HF_TOKEN")

repo_id = "ameer00712/gpt-oss-20b-text2sql-lora"

upload_folder(
    repo_id=repo_id,
    folder_path="/content/outputs/checkpoint-30",
    token=hf_token,
    ignore_patterns=[
        "optimizer.pt",
        "scheduler.pt",
        "rng_state.pth",
        "training_args.bin",
        "trainer_state.json",
    ],
    commit_message="Upload LoRA adapter checkpoint",
)

print("Upload completed:", f"https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint-30/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

  ...adapter_model.safetensors:   0%|          |  412kB /  370MB            

done: https://huggingface.co/ameer00712/gpt-oss-20b-text2sql-lora


To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [34]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="ameer00712/gpt-oss-20b-text2sql-lora",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

SYSTEM = (
    "You are a SQL expert. "
    "Given a database schema and a question, "
    "reply with only the SQL query."
)

schema = (
    "CREATE TABLE machines ("
    "id INT, name TEXT, line TEXT"
    "); "
    "CREATE TABLE breakdowns ("
    "id INT, machine_id INT, downtime_min INT, date DATE"
    ");"
)

question = "Which machine had the most total downtime in 2024?"

messages = [
    {"role": "system", "content": SYSTEM},
    {
        "role": "user",
        "content": f"Database schema:\n{schema}\n\nQuestion: {question}",
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="medium",
).to("cuda")

from transformers import TextStreamer

_ = model.generate(
    **inputs,
    max_new_tokens=512,
    streamer=TextStreamer(tokenizer),
)

### Saving to float16 for VLLM or mxfp4

We also support saving to `float16` or `mxfp4` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("gpt_oss_finetune_4bit", tokenizer, save_method = "mxfp4")
if False: model.push_to_hub_merged("repo_id/gpt_oss_finetune_4bit", tokenizer, token = "YOUR_HF_TOKEN", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")